In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# load files (they are expected to be in the working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# helper to compute summary for metric/trace/log style files
def summarize_timeseries_df(df, name_col):
    total_rows = len(df)
    if total_rows == 0:
        min_ts = None
        max_ts = None
    else:
        times = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        min_ts = times.min().strftime('%Y-%m-%d %H:%M:%S %Z')
        max_ts = times.max().strftime('%Y-%m-%d %H:%M:%S %Z')
    cmdb_sample = df['cmdb_id'].drop_duplicates().head(20).reset_index(drop=True)
    name_sample = df[name_col].drop_duplicates().head(20).reset_index(drop=True)
    summary = pd.DataFrame([{
        'file': name_col.replace('_name','') if name_col.endswith('_name') else name_col,
        'total_rows': total_rows,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'sample_cmdb_ids': [cmdb_sample.tolist()],
        'sample_names': [name_sample.tolist()]
    }])
    return summary

metric_summary = summarize_timeseries_df(metric_df, 'kpi_name')
trace_summary = summarize_timeseries_df(trace_df, 'trace_name')
log_summary = summarize_timeseries_df(log_df, 'log_name')

# error_logs summary: counts, min/max ts, up to first 20 unique cmdb_id, top 5 rows (timestamp, cmdb_id, message)
error_total = len(error_df)
if error_total == 0:
    error_min_ts = None
    error_max_ts = None
else:
    error_times = pd.to_datetime(error_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    error_min_ts = error_times.min().strftime('%Y-%m-%d %H:%M:%S %Z')
    error_max_ts = error_times.max().strftime('%Y-%m-%d %H:%M:%S %Z')

error_cmdb_sample = error_df['cmdb_id'].drop_duplicates().head(20).reset_index(drop=True)

# top 5 rows by timestamp ascending, convert timestamp to timezone-aware string
if error_total > 0:
    err_top5 = error_df.copy()
    err_top5['timestamp'] = pd.to_datetime(err_top5['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    err_top5 = err_top5.sort_values('timestamp').loc[:, ['timestamp', 'cmdb_id', 'message']].head(5).reset_index(drop=True)
else:
    err_top5 = error_df.loc[:, ['timestamp', 'cmdb_id', 'message']].head(0)

error_summary = pd.DataFrame([{
    'file': 'error_logs',
    'total_rows': error_total,
    'min_timestamp': error_min_ts,
    'max_timestamp': error_max_ts,
    'sample_cmdb_ids': [error_cmdb_sample.tolist()],
    'top_5_rows': [err_top5]
}])

# Display compact summaries
metric_summary, trace_summary, log_summary, error_summary

```
Out[1]:
```
```python
# Summarize the previously computed summaries into a compact plain-English string.
m_count = int(metric_summary.at[0, 'total_rows'])
m_min = metric_summary.at[0, 'min_timestamp']
m_max = metric_summary.at[0, 'max_timestamp']
m_cmdbs = metric_summary.at[0, 'sample_cmdb_ids'][0] if metric_summary.at[0, 'sample_cmdb_ids'] else []
m_names = metric_summary.at[0, 'sample_names'][0] if metric_summary.at[0, 'sample_names'] else []

t_count = int(trace_summary.at[0, 'total_rows'])
t_min = trace_summary.at[0, 'min_timestamp']
t_max = trace_summary.at[0, 'max_timestamp']
t_cmdbs = trace_summary.at[0, 'sample_cmdb_ids'][0] if trace_summary.at[0, 'sample_cmdb_ids'] else []
t_names = trace_summary.at[0, 'sample_names'][0] if trace_summary.at[0, 'sample_names'] else []

l_count = int(log_summary.at[0, 'total_rows'])
l_min = log_summary.at[0, 'min_timestamp']
l_max = log_summary.at[0, 'max_timestamp']
l_cmdbs = log_summary.at[0, 'sample_cmdb_ids'][0] if log_summary.at[0, 'sample_cmdb_ids'] else []
l_names = log_summary.at[0, 'sample_names'][0] if log_summary.at[0, 'sample_names'] else []

e_count = int(error_summary.at[0, 'total_rows'])
e_min = error_summary.at[0, 'min_timestamp']
e_max = error_summary.at[0, 'max_timestamp']

# Helper to join up to N examples
def examples(lst, n=6):
    if not lst:
        return "None"
    return ", ".join(lst[:n]) + (", ..." if len(lst) > n else "")

summary = (
    f"metric.csv: {m_count} rows; time range {m_min} to {m_max} (UTC+8). "
    f"Sample cmdb_ids: {examples(m_cmdbs, 8)}. "
    f"Sample KPI names: {examples(m_names, 8)}.\n\n"
    f"trace.csv: {t_count} rows; time range {t_min} to {t_max} (UTC+8). "
    f"Sample cmdb_ids: {examples(t_cmdbs, 8)}. "
    f"Sample trace names: {examples(t_names, 8)}.\n\n"
    f"log.csv: {l_count} rows; time range {l_min} to {l_max} (UTC+8). "
    f"Sample cmdb_ids: {examples(l_cmdbs, 8)}. "
    f"Sample log names: {examples(l_names, 8)}.\n\n"
    f"error_logs.csv: {e_count} rows; no error log entries found in the dataset (min/max timestamps: {e_min}, {e_max})."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  file  total_rows            min_timestamp            max_timestamp                                    sample_cmdb_ids                                       sample_names
0  kpi      319703  2022-03-20 13:00:00 CST  2022-03-20 14:00:00 CST  [[adservice, adservice-0, adservice-1, adservi...  [[app.grpc.count, app.grpc.mrt, app.grpc.rr, a...,     file  total_rows            min_timestamp            max_timestamp                                    sample_cmdb_ids                                       sample_names
0  trace       36040  2022-03-20 13:30:00 CST  2022-03-20 13:59:00 CST  [[adservice-0, adservice-1, adservice-2, adser...  [[trace.from_frontend-0.duration_mean, trace.f...,   file  total_rows            min_timestamp            max_timestamp                                    sample_cmdb_ids                        sample_names
0  log        1844  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST  [[adservice-0, adservice-1, adservice-2, carts...  [[log.error_count, log.row_count]],          file  total_rows min_timestamp max_timestamp sample_cmdb_ids top_5_rows
0  error_logs           0          None          None            [[]]       [[]])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
from datetime import datetime

# reuse metric_df and tz if available in session; otherwise load and set tz
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp column as timezone-aware datetimes in UTC then converted to Asia/Shanghai
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 for each (cmdb_id, kpi_name) using the entire series
global_df = metric_df.groupby(['cmdb_id', 'kpi_name'], observed=True)['value'].agg(
    global_P95=lambda x: x.quantile(0.95),
    global_P5=lambda x: x.quantile(0.05)
).reset_index()

# Total distinct pairs scanned
total_pairs = global_df.shape[0]

# 2) Filter to the incident window (UTC+8) 2022-03-20 13:30:00 to 14:00:00 inclusive
start_local = tz.localize(datetime(2022, 3, 20, 13, 30, 0))
end_local = tz.localize(datetime(2022, 3, 20, 14, 0, 0))

window_mask = (metric_df['ts'] >= start_local) & (metric_df['ts'] <= end_local)
window_rows = metric_df.loc[window_mask].copy()

# Merge global thresholds into window rows to evaluate anomalies against global percentiles
window_rows = window_rows.merge(global_df, on=['cmdb_id', 'kpi_name'], how='left')

# boolean flags for anomalies in the window
window_rows['ge_p95'] = window_rows['value'] >= window_rows['global_P95']
window_rows['le_p5'] = window_rows['value'] <= window_rows['global_P5']

# aggregate within window for pairs that have data in the window
if not window_rows.empty:
    grouped = window_rows.groupby(['cmdb_id', 'kpi_name'], observed=True)
    agg_window = grouped.agg(
        window_points=('value', 'size'),
        anomaly_count_ge_p95=('ge_p95', 'sum'),
        earliest_anom_ts_ge_p95=('ts', lambda s: s[window_rows.loc[s.index, 'ge_p95']].min()),
        anomaly_count_le_p5=('le_p5', 'sum'),
        earliest_anom_ts_le_p5=('ts', lambda s: s[window_rows.loc[s.index, 'le_p5']].min()),
        max_value_in_window=('value', 'max'),
        min_value_in_window=('value', 'min')
    ).reset_index()
else:
    # empty window: create empty dataframe with expected columns
    agg_window = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','window_points','anomaly_count_ge_p95','earliest_anom_ts_ge_p95',
        'anomaly_count_le_p5','earliest_anom_ts_le_p5','max_value_in_window','min_value_in_window'
    ])

# 2b) Merge global thresholds with window aggregates so every global pair is represented
result = global_df.merge(agg_window, on=['cmdb_id', 'kpi_name'], how='left')

# Fill missing numeric counts with 0 and keep min/max as NaN where no window data
result['window_points'] = result['window_points'].fillna(0).astype(int)
result['anomaly_count_ge_p95'] = result['anomaly_count_ge_p95'].fillna(0).astype(int)
result['anomaly_count_le_p5'] = result['anomaly_count_le_p5'].fillna(0).astype(int)

# earliest timestamp columns may be NaT; ensure datetime dtype
for col in ['earliest_anom_ts_ge_p95', 'earliest_anom_ts_le_p5']:
    if col in result.columns:
        result[col] = pd.to_datetime(result[col]).dt.tz_convert(tz)

# compute an overall earliest anomaly timestamp for tie-breaking (min of the two, ignoring NaT)
result['earliest_anom_overall'] = result[['earliest_anom_ts_ge_p95','earliest_anom_ts_le_p5']].min(axis=1)

# compute total anomaly count for ranking
result['total_anomalies'] = result['anomaly_count_ge_p95'] + result['anomaly_count_le_p5']

# 3) Select and format output columns, sort by total anomalies desc, then earliest anomaly asc
result_sorted = result.sort_values(
    by=['total_anomalies','earliest_anom_overall'],
    ascending=[False, True],
    na_position='last'
).reset_index(drop=True)

# Format earliest timestamps as strings (compact) or None
def fmt_ts(s):
    if pd.isna(s):
        return None
    return s.strftime('%Y-%m-%d %H:%M:%S %Z')

out = result_sorted.loc[:, [
    'cmdb_id','kpi_name','global_P95','global_P5','window_points',
    'anomaly_count_ge_p95','earliest_anom_ts_ge_p95',
    'anomaly_count_le_p5','earliest_anom_ts_le_p5',
    'max_value_in_window','min_value_in_window'
]].copy()

out['earliest_anom_ts_ge_p95'] = out['earliest_anom_ts_ge_p95'].apply(fmt_ts)
out['earliest_anom_ts_le_p5'] = out['earliest_anom_ts_le_p5'].apply(fmt_ts)

# Keep only top 20 rows as requested
top20 = out.head(20)

# Compact summary of total pairs scanned
total_pairs_summary = pd.DataFrame([{
    'total_distinct_cmdb_kpi_pairs_scanned': total_pairs
}])

# Return compact table and count
top20, total_pairs_summary

```
Out[2]:
```
```python
summary = (
    "Summary of metric.csv analysis (incident window 2022-03-20 13:30:00 to 14:00:00 UTC+8): "
    "Top 20 (cmdb_id, kpi_name) with most anomalies are all from 'adservice'. "
    "For each of these top rows: global P95 and P5 are identical (e.g., 100.0, 24.0, 5162.0, etc.), "
    "the window contains 31 points (the full 31-minute window), and every point in the window "
    "meets both >= global_P95 and <= global_P5 for those metrics, so anomaly_count_ge_p95 = 31 "
    "and anomaly_count_le_p5 = 31. The earliest anomaly timestamps are 2022-03-20 13:30:00 (UTC+8). "
    "Total distinct (cmdb_id, kpi_name) pairs scanned: 5,253."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                           kpi_name   global_P95    global_P5  window_points  anomaly_count_ge_p95  earliest_anom_ts_ge_p95  anomaly_count_le_p5   earliest_anom_ts_le_p5  max_value_in_window  min_value_in_window
0   adservice                                        app.grpc.rr        100.0        100.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST                100.0                100.0
1   adservice                                        app.grpc.sr        100.0        100.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST                100.0                100.0
2   adservice                                     app.http.count         24.0         24.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST                 24.0                 24.0
3   adservice                                        app.http.rr        100.0        100.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST                100.0                100.0
4   adservice                                        app.http.sr        100.0        100.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST                100.0                100.0
5   adservice    runtime.java_lang_ClassLoading_LoadedClassCount       5162.0       5162.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST               5162.0               5162.0
6   adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...       5185.0       5185.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST               5185.0               5185.0
7   adservice  runtime.java_lang_ClassLoading_UnloadedClassCount         23.0         23.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST                 23.0                 23.0
8   adservice             runtime.java_lang_ClassLoading_Verbose          0.0          0.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST                  0.0                  0.0
9   adservice  runtime.java_lang_Compilation_CompilationTimeM...          1.0          1.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST                  1.0                  1.0
10  adservice  runtime.java_lang_Compilation_TotalCompilation...      47015.0      47015.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST              47015.0              47015.0
11  adservice  runtime.java_lang_GarbageCollector_CollectionC...          8.0          8.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST                  8.0                  8.0
12  adservice  runtime.java_lang_GarbageCollector_CollectionT...        480.0        480.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST                480.0                480.0
13  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...          1.0          1.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST                  1.0                  1.0
14  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...          1.0          1.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST                  1.0                  1.0
15  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...         94.0         94.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST                 94.0                 94.0
16  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  539245384.0  539245384.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST          539245384.0          539245384.0
17  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...          8.0          8.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST                  8.0                  8.0
18  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...   23330816.0   23330816.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST           23330816.0           23330816.0
19  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...   23330816.0   23330816.0             31                    31  2022-03-20 13:30:00 CST                   31  2022-03-20 13:30:00 CST           23330816.0           23330816.0,    total_distinct_cmdb_kpi_pairs_scanned
0                                   5253)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
from datetime import datetime

# reuse trace_df and tz if available; otherwise load and set timezone
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure timezone-aware timestamp column (UTC -> convert to Asia/Shanghai)
trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 for each (cmdb_id, trace_name) using the full series
global_thresh = trace_df.groupby(['cmdb_id', 'trace_name'], observed=True)['value'].agg(
    global_P95=lambda x: x.quantile(0.95),
    global_P5=lambda x: x.quantile(0.05)
).reset_index()

total_pairs = global_thresh.shape[0]

# 2) Filter to incident window (inclusive) in UTC+8
start_local = tz.localize(datetime(2022, 3, 20, 13, 30, 0))
end_local = tz.localize(datetime(2022, 3, 20, 14, 0, 0))

# Merge thresholds into trace_df (thresholds computed globally as required)
trace_with_thresh = trace_df.merge(global_thresh, on=['cmdb_id', 'trace_name'], how='left')

window_mask = (trace_with_thresh['ts'] >= start_local) & (trace_with_thresh['ts'] <= end_local)
window_rows = trace_with_thresh.loc[window_mask].copy()

# If window has data, compute anomaly flags and aggregates; otherwise prepare empty structures
if not window_rows.empty:
    window_rows['ge_p95'] = window_rows['value'] >= window_rows['global_P95']
    window_rows['le_p5'] = window_rows['value'] <= window_rows['global_P5']

    grouped = window_rows.groupby(['cmdb_id', 'trace_name'], observed=True)

    agg_counts = grouped.agg(
        window_points=('value', 'size'),
        anomaly_count_ge_p95=('ge_p95', 'sum'),
        anomaly_count_le_p5=('le_p5', 'sum'),
        max_value_in_window=('value', 'max'),
        min_value_in_window=('value', 'min')
    ).reset_index()

    # earliest anomaly timestamps (may be NaT)
    earliest_ge = grouped.apply(lambda g: g.loc[g['ge_p95'], 'ts'].min()).reset_index().rename(columns={0: 'earliest_anom_ts_ge_p95'})
    earliest_le = grouped.apply(lambda g: g.loc[g['le_p5'], 'ts'].min()).reset_index().rename(columns={0: 'earliest_anom_ts_le_p5'})

    agg_window = agg_counts.merge(earliest_ge, on=['cmdb_id', 'trace_name'], how='left').merge(earliest_le, on=['cmdb_id', 'trace_name'], how='left')
else:
    agg_window = pd.DataFrame(columns=[
        'cmdb_id','trace_name','window_points','anomaly_count_ge_p95','anomaly_count_le_p5',
        'max_value_in_window','min_value_in_window','earliest_anom_ts_ge_p95','earliest_anom_ts_le_p5'
    ])

# 2b) Merge global thresholds with window aggregates so every global pair is represented
result = global_thresh.merge(agg_window, on=['cmdb_id', 'trace_name'], how='left')

# Fill numeric NaNs with 0 and convert to int where appropriate
result['window_points'] = result['window_points'].fillna(0).astype(int)
result['anomaly_count_ge_p95'] = result['anomaly_count_ge_p95'].fillna(0).astype(int)
result['anomaly_count_le_p5'] = result['anomaly_count_le_p5'].fillna(0).astype(int)

# earliest timestamps may be NaT; ensure tz-aware dtype where present
for col in ['earliest_anom_ts_ge_p95', 'earliest_anom_ts_le_p5']:
    if col in result.columns:
        result[col] = pd.to_datetime(result[col])

# compute tie-breaker earliest anomaly overall (min of the two, ignoring NaT)
result['earliest_anom_overall'] = result[['earliest_anom_ts_ge_p95', 'earliest_anom_ts_le_p5']].min(axis=1)

# total anomalies for ranking
result['total_anomalies'] = result['anomaly_count_ge_p95'] + result['anomaly_count_le_p5']

# sort by total_anomalies desc, then earliest_anom_overall asc (NaT go last)
result_sorted = result.sort_values(by=['total_anomalies', 'earliest_anom_overall'], ascending=[False, True], na_position='last').reset_index(drop=True)

# format earliest timestamps to compact strings or None
def fmt_ts(val):
    if pd.isna(val):
        return None
    # val is tz-aware already in trace_df; ensure converted to Asia/Shanghai and formatted
    try:
        return val.tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return pd.to_datetime(val).tz_localize(tz, ambiguous='NaT').strftime('%Y-%m-%d %H:%M:%S %Z')

out = result_sorted.loc[:, [
    'cmdb_id','trace_name','global_P95','global_P5','window_points',
    'anomaly_count_ge_p95','earliest_anom_ts_ge_p95',
    'anomaly_count_le_p5','earliest_anom_ts_le_p5',
    'max_value_in_window','min_value_in_window'
]].copy()

out['earliest_anom_ts_ge_p95'] = out['earliest_anom_ts_ge_p95'].apply(fmt_ts)
out['earliest_anom_ts_le_p5'] = out['earliest_anom_ts_le_p5'].apply(fmt_ts)

# limit to top 20 rows for compactness
top20_traces = out.head(20)

# summary of total distinct pairs scanned
total_pairs_summary = pd.DataFrame([{'total_distinct_cmdb_trace_pairs_scanned': total_pairs}])

# Return the compact top-20 table and the total pairs count
top20_traces, total_pairs_summary

```
Out[3]:
```
```python
# Plain-English summary string based on the trace.csv analysis results shown earlier.
summary = (
    "Trace analysis summary (incident window 2022-03-20 13:30:00 to 14:00:00 UTC+8):\n"
    "- Total distinct (cmdb_id, trace_name) pairs scanned: 1,232.\n"
    "- Top 20 pairs (returned) are dominated by per-pod trace error_rate and related metrics\n"
    "  (examples: trace.from_frontend-*.error_rate, trace.from_checkoutservice-*.error_rate, trace.self.*).\n"
    "- For each of these top 20 rows:\n"
    "  • global P95 = 0.0 and global P5 = 0.0 (computed over the full series).\n"
    "  • The incident window contains 30 points for each pair.\n"
    "  • anomaly_count_ge_p95 = 30 and anomaly_count_le_p5 = 30, i.e. every point in the window\n"
    "    equals the global P95 and P5 threshold, so all samples are flagged by both checks.\n"
    "  • Earliest anomaly timestamps are 2022-03-20 13:30:00 (UTC+8).\n"
    "  • max and min values in the window are 0.0 for these rows.\n"
    "\n"
    "Interpretation: many trace error_rate / related trace metrics are constant at 0.0 across both\n"
    "their full history (giving P5=P95=0.0) and the incident window, which causes every point in\n"
    "the window to match both percentile thresholds. This leads to high anomaly counts by the\n"
    "simple >=P95 / <=P5 rule even though the value is simply constant at zero."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id                               trace_name  global_P95  global_P5  window_points  anomaly_count_ge_p95  earliest_anom_ts_ge_p95  anomaly_count_le_p5   earliest_anom_ts_le_p5  max_value_in_window  min_value_in_window
0     adservice-0         trace.from_frontend-0.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
1     adservice-0         trace.from_frontend-1.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
2     adservice-0         trace.from_frontend-2.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
3     adservice-1         trace.from_frontend-0.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
4     adservice-1         trace.from_frontend-1.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
5     adservice-1         trace.from_frontend-2.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
6     adservice-2         trace.from_frontend-0.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
7     adservice-2         trace.from_frontend-1.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
8     adservice-2         trace.from_frontend-2.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
9    adservice2-0        trace.from_frontend2-0.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
10  cartservice-0  trace.from_checkoutservice-0.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
11  cartservice-0  trace.from_checkoutservice-1.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
12  cartservice-0  trace.from_checkoutservice-2.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
13  cartservice-0         trace.from_frontend-0.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
14  cartservice-0         trace.from_frontend-1.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
15  cartservice-0         trace.from_frontend-2.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
16  cartservice-0                  trace.self.duration_p95         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
17  cartservice-0                    trace.self.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
18  cartservice-1  trace.from_checkoutservice-0.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0
19  cartservice-1  trace.from_checkoutservice-1.error_rate         0.0        0.0             30                    30  2022-03-20 13:30:00 CST                   30  2022-03-20 13:30:00 CST                  0.0                  0.0,    total_distinct_cmdb_trace_pairs_scanned
0                                     1232)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
from datetime import datetime
import numpy as np

# Reuse metric_df and tz if present, otherwise load and set timezone to UTC+8
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure tz-aware timestamp column (convert from epoch seconds -> UTC then to Asia/Shanghai)
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz aware and converted
    metric_df['ts'] = pd.to_datetime(metric_df['ts']).dt.tz_localize('UTC')\
        .dt.tz_convert(tz) if metric_df['ts'].dtype == object else metric_df['ts'].dt.tz_convert(tz)

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using full series (required)
global_thresh = metric_df.groupby(['cmdb_id', 'kpi_name'], observed=True)['value'].agg(
    global_P95=lambda x: x.quantile(0.95),
    global_P5=lambda x: x.quantile(0.05)
).reset_index()

total_pairs_scanned = global_thresh.shape[0]

# Merge thresholds into metric_df (so thresholds computed globally are available per row)
metric_with_thresh = metric_df.merge(global_thresh, on=['cmdb_id', 'kpi_name'], how='left')

# 2) Filter to incident window (inclusive) in UTC+8
start_local = tz.localize(datetime(2022, 3, 20, 13, 30, 0))
end_local = tz.localize(datetime(2022, 3, 20, 14, 0, 0))

window_mask = (metric_with_thresh['ts'] >= start_local) & (metric_with_thresh['ts'] <= end_local)
window_df = metric_with_thresh.loc[window_mask].copy()

# Identify rows in window where value >= global_P95 OR value <= global_P5
window_df['ge_p95'] = window_df['value'] >= window_df['global_P95']
window_df['le_p5'] = window_df['value'] <= window_df['global_P5']
anomaly_rows = window_df[window_df['ge_p95'] | window_df['le_p5']].copy()

# 2b) For each (cmdb_id, kpi_name), group contiguous timestamps into consecutive-minute runs
fault_records = []
if not anomaly_rows.empty:
    # sort for deterministic processing
    anomaly_rows = anomaly_rows.sort_values(['cmdb_id', 'kpi_name', 'ts']).reset_index(drop=True)

    # Process group by group
    for (cmdb, kpi), grp in anomaly_rows.groupby(['cmdb_id', 'kpi_name'], sort=False):
        grp = grp.sort_values('ts').reset_index(drop=True)
        # compute gap in seconds between consecutive points
        gaps = grp['ts'].diff().dt.total_seconds().fillna(0)
        # run id increments when gap != 60 seconds (consecutive minute required)
        run_id = (gaps != 60).cumsum()

        grp['run_id'] = run_id

        for rid, run in grp.groupby('run_id', sort=True):
            run = run.sort_values('ts')
            start_ts = run['ts'].iloc[0]
            end_ts = run['ts'].iloc[-1]
            length_minutes = int(run.shape[0])
            count_ge = int(run['ge_p95'].sum())
            count_le = int(run['le_p5'].sum())
            # decide fault_type: majority; if tie choose "high"
            if count_ge > count_le:
                fault_type = 'high'
            elif count_le > count_ge:
                fault_type = 'low'
            else:
                fault_type = 'high'
            max_val = float(run['value'].max())
            min_val = float(run['value'].min())
            global_p95 = float(run['global_P95'].iloc[0])
            global_p5 = float(run['global_P5'].iloc[0])
            # severity score formula with denom max(1, threshold)
            if fault_type == 'high':
                denom = max(1.0, global_p95)
                severity = (max_val - global_p95) / denom
            else:
                denom = max(1.0, global_p5)
                severity = (global_p5 - min_val) / denom
            # earliest anomaly type-specific timestamp for run majority (for informational tie-breaks if needed)
            earliest_ge_ts = run.loc[run['ge_p95'], 'ts'].min() if run['ge_p95'].any() else pd.NaT
            earliest_le_ts = run.loc[run['le_p5'], 'ts'].min() if run['le_p5'].any() else pd.NaT

            fault_records.append({
                'cmdb_id': cmdb,
                'kpi_name': kpi,
                'fault_type': fault_type,
                'start_ts': start_ts,
                'end_ts': end_ts,
                'length_minutes': length_minutes,
                'count_points': length_minutes,
                'count_ge_p95': count_ge,
                'count_le_p5': count_le,
                'max_value_in_run': max_val,
                'min_value_in_run': min_val,
                'global_P95': global_p95,
                'global_P5': global_p5,
                'severity_score': float(severity),
                'earliest_ge_ts': earliest_ge_ts,
                'earliest_le_ts': earliest_le_ts
            })

# Build faults DataFrame
faults_df = pd.DataFrame(fault_records)

# If empty, prepare empty DataFrame with expected columns
expected_cols = [
    'cmdb_id','kpi_name','fault_type','start_ts','end_ts','length_minutes','count_points',
    'count_ge_p95','count_le_p5','max_value_in_run','min_value_in_run',
    'global_P95','global_P5','severity_score','earliest_ge_ts','earliest_le_ts'
]
for c in expected_cols:
    if c not in faults_df.columns:
        faults_df[c] = pd.Series(dtype='float64') if c in ['max_value_in_run','min_value_in_run','global_P95','global_P5','severity_score'] \
                        else pd.Series(dtype='object')

# 4) Format timestamps to strings in UTC+8 for display and sort by start_timestamp ascending
if not faults_df.empty:
    faults_df['start_ts_str'] = faults_df['start_ts'].dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    faults_df['end_ts_str'] = faults_df['end_ts'].dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # sort by start timestamp ascending
    faults_df = faults_df.sort_values('start_ts').reset_index(drop=True)
else:
    faults_df['start_ts_str'] = []
    faults_df['end_ts_str'] = []

# Keep compact columns for output
out_cols = [
    'cmdb_id','kpi_name','fault_type','start_ts_str','end_ts_str','length_minutes','count_points',
    'max_value_in_run','min_value_in_run','global_P95','global_P5','severity_score'
]
compact_faults = faults_df.loc[:, out_cols].head(20).copy()

# Aggregate summary: total faults and top 10 cmdb_ids by number of faults
total_faults = int(faults_df.shape[0])
if total_faults > 0:
    top_cmdb = faults_df.groupby('cmdb_id', observed=True).size().reset_index(name='fault_count').sort_values('fault_count', ascending=False).head(10).reset_index(drop=True)
else:
    top_cmdb = pd.DataFrame(columns=['cmdb_id','fault_count'])

total_summary = pd.DataFrame([{
    'total_faults_detected': total_faults,
    'total_distinct_cmdb_kpi_pairs_scanned': total_pairs_scanned
}])

# Display compact results: top fault rows (<=20), total summary, and top 10 components by fault count
compact_faults, total_summary, top_cmdb

```
Out[4]:
```
```python
# Plain-English summary of the fault extraction results (incident window 2022-03-20 13:30:00 to 14:00:00 UTC+8)
summary = (
    "Fault extraction summary:\n"
    "- Total faults detected: 9,429 across all (cmdb_id, kpi_name) pairs.\n"
    "- Total distinct (cmdb_id, kpi_name) pairs scanned for global thresholds: 5,253.\n"
    "- Many detected faults are full-window runs (length_minutes = 31) starting at 2022-03-20 13:30:00 and ending at 14:00:00 UTC+8.\n"
    "- Severity scores for most reported faults are 0.0 (because run values equal the computed global P95/P5 thresholds), with very few small non-zero severity cases (example: one 1-minute low run on productcatalogservice2-0 with severity ~0.0114).\n"
    "- Example entries from the top-20 faults (earliest first) include runs on productcatalogservice2-0, checkoutservice-1, frontend-1, paymentservice-2, recommendationservice and others; many are 'high' type runs of length 31 minutes.\n"
    "- Top 10 components by number of faults:\n"
    "    1) adservice: 418 faults\n"
    "    2) adservice2: 393 faults\n"
    "    3) frontend-1: 384 faults\n"
    "    4) frontend-0: 373 faults\n"
    "    5) frontend2-0: 372 faults\n"
    "    6) frontend-2: 322 faults\n"
    "    7) checkoutservice-1: 315 faults\n"
    "    8) checkoutservice-2: 303 faults\n"
    "    9) checkoutservice2-0: 287 faults\n"
    "   10) checkoutservice-0: 280 faults\n\n"
    "Interpretation notes: many metrics have constant or identical full-series percentiles (P5 == P95 or constant values),"
    " which makes the simple >=P95 / <=P5 rule mark many points as anomalies (often every point in the window),"
    " resulting in large counts of fault runs with zero computed severity. Consider refining thresholds or anomaly rules to avoid flagging constant-valued series."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id                                           kpi_name fault_type             start_ts_str               end_ts_str  length_minutes  count_points  max_value_in_run  min_value_in_run  global_P95  global_P5  severity_score
0   productcatalogservice2-0      container.node-6.container_fs_reads./dev/vda1       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31              0.00              0.00        0.00        0.0        0.000000
1          checkoutservice-1  mesh.source.checkoutservice.productcatalogserv...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31              0.00              0.00        0.00        0.0        0.000000
2          checkoutservice-1  mesh.source.checkoutservice.productcatalogserv...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31              0.00              0.00        0.00        0.0        0.000000
3                 frontend-1  mesh.source.frontend.cartservice.istio_request...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31              0.00              0.00        0.00        0.0        0.000000
4           paymentservice-2    container.node-6.container_cpu_load_average_10s       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31              0.00              0.00        0.00        0.0        0.000000
5           paymentservice-2      container.node-6.container_cpu_system_seconds        low  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31              0.01              0.00        0.01        0.0        0.000000
6          checkoutservice-1  mesh.source.checkoutservice.productcatalogserv...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31              0.00              0.00        0.00        0.0        0.000000
7           paymentservice-2        container.node-6.container_cpu_user_seconds        low  2022-03-20 13:30:00 CST  2022-03-20 13:32:00 CST               3             3              0.01              0.00        0.01        0.0        0.000000
8                 frontend-1  mesh.source.frontend.cartservice.istio_request...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31              0.00              0.00        0.00        0.0        0.000000
9           paymentservice-2        container.node-6.container_file_descriptors       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31             22.00             22.00       22.00       22.0        0.000000
10          paymentservice-2     container.node-6.container_fs_inodes./dev/vda1       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31              0.00              0.00        0.00        0.0        0.000000
11         checkoutservice-1  mesh.source.checkoutservice.productcatalogserv...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31              0.00              0.00        0.00        0.0        0.000000
12          paymentservice-2  container.node-6.container_fs_inodes_free./dev...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31              0.00              0.00        0.00        0.0        0.000000
13     recommendationservice                                        app.grpc.rr       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31            100.00            100.00      100.00      100.0        0.000000
14  productcatalogservice2-0  mesh.source.productcatalogservice2.jaeger-coll...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31              0.00              0.00        0.00        0.0        0.000000
15          paymentservice-2  container.node-6.container_fs_io_current./dev/...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31              0.00              0.00        0.00        0.0        0.000000
16         checkoutservice-1  mesh.source.checkoutservice.productcatalogserv...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31              0.00              0.00        0.00        0.0        0.000000
17         checkoutservice-1  mesh.source.checkoutservice.productcatalogserv...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31              0.00              0.00        0.00        0.0        0.000000
18  productcatalogservice2-0  mesh.source.productcatalogservice2.jaeger-coll...        low  2022-03-20 13:30:00 CST  2022-03-20 13:30:00 CST               1             1           4328.25           4328.25     5174.00     4378.0        0.011364
19  productcatalogservice2-0  mesh.source.productcatalogservice2.jaeger-coll...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31            31              0.00              0.00        0.00        0.0        0.000000,    total_faults_detected  total_distinct_cmdb_kpi_pairs_scanned
0                   9429                                   5253,               cmdb_id  fault_count
0           adservice          418
1          adservice2          393
2          frontend-1          384
3          frontend-0          373
4         frontend2-0          372
5          frontend-2          322
6   checkoutservice-1          315
7   checkoutservice-2          303
8  checkoutservice2-0          287
9   checkoutservice-0          280)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing metric_df and tz if present; otherwise load and set timezone to UTC+8
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timezone-aware timestamp column (epoch seconds -> UTC -> Asia/Shanghai)
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # If ts exists, ensure tz-aware and converted to Asia/Shanghai
    metric_df['ts'] = pd.to_datetime(metric_df['ts'], utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using the full series
global_thresh = metric_df.groupby(['cmdb_id', 'kpi_name'], observed=True)['value'].agg(
    global_P95=lambda x: x.quantile(0.95),
    global_P5=lambda x: x.quantile(0.05)
).reset_index()

# 2) Merge thresholds into metric_df and filter to the incident window (inclusive)
metric_with_thresh = metric_df.merge(global_thresh, on=['cmdb_id', 'kpi_name'], how='left')

start_local = tz.localize(datetime(2022, 3, 20, 13, 30, 0))
end_local = tz.localize(datetime(2022, 3, 20, 14, 0, 0))
window_mask = (metric_with_thresh['ts'] >= start_local) & (metric_with_thresh['ts'] <= end_local)
window_df = metric_with_thresh.loc[window_mask].copy()

# Identify anomaly points (value >= P95 OR value <= P5)
window_df['ge_p95'] = window_df['value'] >= window_df['global_P95']
window_df['le_p5'] = window_df['value'] <= window_df['global_P5']
anomaly_rows = window_df[window_df['ge_p95'] | window_df['le_p5']].copy()

# 2b & 3) Group consecutive-minute runs per (cmdb_id, kpi_name), compute run stats and severity
fault_records = []
if not anomaly_rows.empty:
    anomaly_rows = anomaly_rows.sort_values(['cmdb_id', 'kpi_name', 'ts']).reset_index(drop=True)
    for (cmdb, kpi), grp in anomaly_rows.groupby(['cmdb_id', 'kpi_name'], sort=False):
        grp = grp.sort_values('ts').reset_index(drop=True)
        # compute gaps in seconds; consecutive-minute means diff == 60
        gaps = grp['ts'].diff().dt.total_seconds().fillna(0)
        run_id = (gaps != 60).cumsum()
        grp['run_id'] = run_id
        for rid, run in grp.groupby('run_id', sort=True):
            run = run.sort_values('ts')
            start_ts = run['ts'].iloc[0]
            end_ts = run['ts'].iloc[-1]
            length_minutes = int(run.shape[0])
            max_val = float(run['value'].max())
            min_val = float(run['value'].min())
            global_p95 = float(run['global_P95'].iloc[0])
            global_p5 = float(run['global_P5'].iloc[0])
            count_ge = int(run['ge_p95'].sum())
            count_le = int(run['le_p5'].sum())
            # Determine fault_type by majority, tie -> 'high'
            if count_ge >= count_le:
                fault_type = 'high'
                denom = max(1.0, global_p95)
                severity = (max_val - global_p95) / denom
            else:
                fault_type = 'low'
                denom = max(1.0, global_p5)
                severity = (global_p5 - min_val) / denom
            fault_records.append({
                'cmdb_id': cmdb,
                'kpi_name': kpi,
                'fault_type': fault_type,
                'start_ts': start_ts,
                'end_ts': end_ts,
                'length_minutes': length_minutes,
                'max_value_in_run': max_val,
                'min_value_in_run': min_val,
                'global_P95': global_p95,
                'global_P5': global_p5,
                'severity_score': float(severity)
            })

# Build DataFrame of faults
faults_df = pd.DataFrame(fault_records)
# If empty, create columns to keep consistent display
expected_cols = ['cmdb_id','kpi_name','fault_type','start_ts','end_ts','length_minutes',
                 'max_value_in_run','min_value_in_run','global_P95','global_P5','severity_score']
for c in expected_cols:
    if c not in faults_df.columns:
        faults_df[c] = pd.Series(dtype='object')

# Format timestamps and sort by severity descending for ranking
if not faults_df.empty:
    faults_df['start_ts_dt'] = pd.to_datetime(faults_df['start_ts']).dt.tz_convert(tz)
    faults_df['end_ts_dt'] = pd.to_datetime(faults_df['end_ts']).dt.tz_convert(tz)
    faults_df['start_ts_str'] = faults_df['start_ts_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    faults_df['end_ts_str'] = faults_df['end_ts_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # Rank by severity descending, tie-breaker: earlier start_ts first
    faults_ranked = faults_df.sort_values(by=['severity_score','start_ts_dt'], ascending=[False, True]).reset_index(drop=True)
else:
    faults_ranked = faults_df.copy()

# Select top 20 fault runs with requested columns (compact)
top20_faults = faults_ranked.loc[:, [
    'cmdb_id','kpi_name','fault_type','start_ts_str','end_ts_str','length_minutes',
    'max_value_in_run','min_value_in_run','global_P95','global_P5','severity_score'
]].head(20).copy()

# 4) Component aggregate: top 10 cmdb_ids by sum of severity_score, also earliest fault start per component
if not faults_df.empty:
    comp_agg = faults_df.groupby('cmdb_id', observed=True).agg(
        total_severity=('severity_score', 'sum'),
        earliest_fault_start_dt=('start_ts', 'min')
    ).reset_index()
    # convert earliest timestamp to tz-aware string
    comp_agg['earliest_fault_start_dt'] = pd.to_datetime(comp_agg['earliest_fault_start_dt']).dt.tz_convert(tz)
    comp_agg['earliest_fault_start'] = comp_agg['earliest_fault_start_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    comp_agg_sorted = comp_agg.sort_values(by='total_severity', ascending=False).head(10).loc[:, ['cmdb_id','total_severity','earliest_fault_start']].reset_index(drop=True)
else:
    comp_agg_sorted = pd.DataFrame(columns=['cmdb_id','total_severity','earliest_fault_start'])

# Final compact outputs: top20_faults (<=20 rows) and comp_agg_sorted (<=10 rows)
top20_faults, comp_agg_sorted

```
Out[5]:
```
# Plain-English summary of the recomputed fault runs (ranked by severity, incident window 2022-03-20 13:30─14:00 UTC+8)
summary = (
    "Recomputed fault runs using full-series P95/P5 and focused on real deviations (window 2022-03-20 13:30:00–14:00:00 UTC+8):\n"
    "- Top fault runs by severity (selected examples):\n"
    "  • node-5: system.io.rkb_s — highest severity (64.0), 31-minute run starting 2022-03-20 13:30.\n"
    "  • shippingservice-0: several container.memory_failures metrics — severity ~33, 31-minute runs starting 13:30.\n"
    "  • node-6: system.io.rkb_s — severity 14.0, 31-minute run.\n"
    "  • node-1: spikes on system.io.w_s and system.cpu.iowait — single-minute or short runs with severity ~6.9–8.2.\n"
    "  • node-4 and others show elevated CPU metrics (severity ~4–8) and multiple 1–3 minute runs.\n"
    "- Aggregated by component (top 10 by total severity sum):\n"
    "  1) shippingservice-0 (total_severity ~68.02), 2) node-5 (~67.72), 3) node-1 (~35.30), 4) node-6 (~19.23),\n"
    "  5) node-4 (~16.73), 6) checkoutservice2-0 (~14.06), 7) cartservice2-0 (~13.45), 8) checkoutservice-0 (~12.97),\n"
    "  9) paymentservice-0 (~12.50), 10) frontend-1 (~10.54). Earliest faults for these components begin at 2022-03-20 13:30:00 UTC+8.\n"
    "- Interpretation: node-level I/O and some container memory-failure metrics (notably on shippingservice-0 and node-5) dominate severity. "
    "Short, high-magnitude single-minute spikes (node-1 I/O writes, CPU iowait) also contribute significant severity.\n"
    "- Recommended next steps: investigate node-5 and shippingservice-0 I/O/memory metrics and the node-1 write/iowait spikes as primary leads."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                           kpi_name fault_type             start_ts_str               end_ts_str  length_minutes  max_value_in_run  min_value_in_run  global_P95  global_P5  severity_score
0              node-5                                    system.io.rkb_s       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31             64.00              0.00        0.00       0.00       64.000000
1   shippingservice-0  container.node-6.container_memory_failures.con...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31             33.00              0.00        0.00       0.00       33.000000
2   shippingservice-0  container.node-6.container_memory_failures.hie...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31             33.00              0.00        0.00       0.00       33.000000
3              node-6                                    system.io.rkb_s       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31             14.00              0.00        0.00       0.00       14.000000
4              node-1                                      system.io.w_s       high  2022-03-20 13:54:00 CST  2022-03-20 13:54:00 CST               1            314.00            314.00       34.00       0.00        8.235294
5              node-4                                    system.cpu.user       high  2022-03-20 13:40:00 CST  2022-03-20 13:42:00 CST               3             55.18             54.55        6.00       1.06        8.196667
6              node-1                                      system.io.w_s       high  2022-03-20 13:47:00 CST  2022-03-20 13:47:00 CST               1            269.50            269.50       34.00       0.00        6.926471
7              node-1                                  system.cpu.iowait       high  2022-03-20 13:30:00 CST  2022-03-20 13:30:00 CST               1              7.02              7.02        1.15       0.17        5.104348
8              node-4                               system.cpu.pct_usage       high  2022-03-20 13:40:00 CST  2022-03-20 13:42:00 CST               3             60.03             57.48       11.35       3.27        4.288987
9      cartservice2-0       container.node-5.container_fs_reads./dev/vda       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31              4.00              0.00        0.00       0.00        4.000000
10   paymentservice-0  container.node-6.container_memory_failures.con...       high  2022-03-20 14:00:00 CST  2022-03-20 14:00:00 CST               1            396.00            396.00       82.50       0.00        3.800000
11   paymentservice-0  container.node-6.container_memory_failures.hie...       high  2022-03-20 14:00:00 CST  2022-03-20 14:00:00 CST               1            396.00            396.00       82.50       0.00        3.800000
12             node-1                        system.net.tcp.retrans_segs       high  2022-03-20 13:30:00 CST  2022-03-20 13:32:00 CST               3              3.83              0.00        0.05       0.00        3.780000
13             node-6                                      system.io.r_s       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31              3.50              0.00        0.00       0.00        3.500000
14  checkoutservice-2       container.node-5.container_fs_reads./dev/vda       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31              3.00              0.00        0.00       0.00        3.000000
15  shippingservice-2       container.node-5.container_fs_reads./dev/vda       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31              3.00              0.00        0.00       0.00        3.000000
16   paymentservice-1  container.node-6.container_memory_failures.con...       high  2022-03-20 13:39:00 CST  2022-03-20 13:40:00 CST               2            379.50            330.00      115.50       0.00        2.285714
17   paymentservice-1  container.node-6.container_memory_failures.hie...       high  2022-03-20 13:39:00 CST  2022-03-20 13:40:00 CST               2            379.50            330.00      115.50       0.00        2.285714
18      cartservice-1  container.node-6.container_cpu_cfs_throttled_p...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31              2.00              0.00        0.00       0.00        2.000000
19     cartservice2-0  container.node-5.container_cpu_cfs_throttled_p...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31              2.00              0.00        0.00       0.00        2.000000,               cmdb_id  total_severity     earliest_fault_start
0   shippingservice-0       68.022277  2022-03-20 13:30:00 CST
1              node-5       67.717956  2022-03-20 13:30:00 CST
2              node-1       35.302963  2022-03-20 13:30:00 CST
3              node-6       19.226776  2022-03-20 13:30:00 CST
4              node-4       16.730478  2022-03-20 13:30:00 CST
5  checkoutservice2-0       14.056562  2022-03-20 13:30:00 CST
6      cartservice2-0       13.448127  2022-03-20 13:30:00 CST
7   checkoutservice-0       12.974047  2022-03-20 13:30:00 CST
8    paymentservice-0       12.497373  2022-03-20 13:30:00 CST
9          frontend-1       10.540129  2022-03-20 13:30:00 CST)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse variables from kernel if available; otherwise load files
try:
    faults_df
except NameError:
    faults_df = pd.read_csv('metric_faults.csv')  # fallback (unlikely)

try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure faults_df exists and has severity_score and timestamps; if start_ts is not tz-aware, try to parse
if 'faults_df' not in globals():
    faults_df = pd.DataFrame()  # safe empty

# 1) Fault rows for node-5 or shippingservice-0 with severity_score > 0
if not faults_df.empty:
    # Ensure start_ts/end_ts are datetimes and tz-aware
    if faults_df['start_ts'].dtype == object or not pd.api.types.is_datetime64_any_dtype(faults_df['start_ts']):
        faults_df['start_ts'] = pd.to_datetime(faults_df['start_ts']).dt.tz_localize('UTC', ambiguous='NaT', nonexistent='NaT').dt.tz_convert(tz)
    else:
        faults_df['start_ts'] = pd.to_datetime(faults_df['start_ts']).dt.tz_convert(tz)
    if 'end_ts' in faults_df.columns:
        if faults_df['end_ts'].dtype == object or not pd.api.types.is_datetime64_any_dtype(faults_df['end_ts']):
            faults_df['end_ts'] = pd.to_datetime(faults_df['end_ts']).dt.tz_localize('UTC', ambiguous='NaT', nonexistent='NaT').dt.tz_convert(tz)
        else:
            faults_df['end_ts'] = pd.to_datetime(faults_df['end_ts']).dt.tz_convert(tz)

    filt = faults_df[
        faults_df['cmdb_id'].isin(['node-5', 'shippingservice-0']) &
        (faults_df['severity_score'].astype(float) > 0)
    ].copy()

    # Format timestamps to strings
    if not filt.empty:
        filt['start_timestamp'] = filt['start_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        filt['end_timestamp'] = filt['end_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

    node_shipping_faults = filt.loc[:, [
        'cmdb_id','kpi_name','fault_type','start_timestamp','end_timestamp',
        'length_minutes','max_value_in_run','min_value_in_run','global_P95','global_P5','severity_score'
    ]].sort_values(by='severity_score', ascending=False).head(20).reset_index(drop=True)
else:
    node_shipping_faults = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_type','start_timestamp','end_timestamp',
        'length_minutes','max_value_in_run','min_value_in_run','global_P95','global_P5','severity_score'
    ])

# 2) Trace rows for shippingservice-related traces in the incident window, with global P95/P5
# Compute global thresholds per (cmdb_id, trace_name) using full trace series
if trace_df.empty:
    trace_hits = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','global_P95','global_P5'])
else:
    # ensure tz-aware ts column
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

    trace_global = trace_df.groupby(['cmdb_id','trace_name'], observed=True)['value'].agg(
        global_P95=lambda x: x.quantile(0.95),
        global_P5=lambda x: x.quantile(0.05)
    ).reset_index()

    # Filter to incident window
    start_local = tz.localize(datetime(2022,3,20,13,30,0))
    end_local = tz.localize(datetime(2022,3,20,14,0,0))
    mask_window = (trace_df['ts'] >= start_local) & (trace_df['ts'] <= end_local)

    # Filter to shippingservice patterns
    mask_cmdb = trace_df['cmdb_id'].astype(str).str.contains('shippingservice', na=False)
    mask_name = trace_df['trace_name'].astype(str).str.contains('shippingservice', na=False)
    mask_target = mask_cmdb | mask_name

    trace_filtered = trace_df.loc[mask_window & mask_target].copy()

    if not trace_filtered.empty:
        # merge global thresholds
        trace_filtered = trace_filtered.merge(trace_global, on=['cmdb_id','trace_name'], how='left')
        trace_filtered['timestamp_str'] = trace_filtered['ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        trace_hits = trace_filtered.loc[:, ['timestamp_str','cmdb_id','trace_name','value','global_P95','global_P5']].rename(columns={'timestamp_str':'timestamp'}).head(20).reset_index(drop=True)
    else:
        trace_hits = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','global_P95','global_P5'])

# 3) From log.csv, filter to window and components node-5, node-6, shippingservice-0
if log_df.empty:
    log_hits = pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value'])
else:
    log_df['ts'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    start_local = tz.localize(datetime(2022,3,20,13,30,0))
    end_local = tz.localize(datetime(2022,3,20,14,0,0))
    mask_window_log = (log_df['ts'] >= start_local) & (log_df['ts'] <= end_local)
    mask_cmdb_log = log_df['cmdb_id'].isin(['shippingservice-0','node-5','node-6'])
    log_filtered = log_df.loc[mask_window_log & mask_cmdb_log].copy()
    if not log_filtered.empty:
        log_filtered['timestamp'] = log_filtered['ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        log_hits = log_filtered.loc[:, ['timestamp','cmdb_id','log_name','value']].head(20).reset_index(drop=True)
    else:
        log_hits = pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value'])

# error_logs row count
error_count = len(error_df) if not error_df is None else 0
error_count_df = pd.DataFrame([{'error_logs_row_count': error_count}])

# Return the three compact tables and the error count
node_shipping_faults, trace_hits, log_hits, error_count_df

```
Out[6]:
```
```python
# Plain-English compact summary based on the returned tables for node-5 and shippingservice-0
summary = (
    "Summary for node-5 and shippingservice-0 (incident window 2022-03-20 13:30–14:00 UTC+8):\n"
    "- Fault runs (severity>0) found:\n"
    "  • node-5: multiple faults including a dominant system.io.rkb_s 31-minute run (severity 64.0),\n"
    "    additional shorter runs for I/O and CPU (load, r_s, r_await, w_await) with smaller severities.\n"
    "  • shippingservice-0: several high-severity container memory-failure runs (~33.0) and network-related\n"
    "    spikes (packet/byte counts) with non-zero severity (examples shown around 0.09–0.66 for single-minute network runs).\n"
    "- Trace signals (shipping-related): up to 20 trace rows returned show calls to/from shippingservice with very small\n"
    "  duration values (e.g., duration_mean/p95 ~0.00006–0.0001) and zero or near-zero error_rate; row_count and throughput\n"
    "  examples are small integers (row_count typically 2–4 in the sample). Global P95/P5 for these traces are also very small\n"
    "  (consistent with the low values observed).\n"
    "- Logs for shippingservice-0 (window sample): log.error_count is 0 for all returned minutes; log.row_count fluctuates\n"
    "  around ~95–122 per minute in the shown samples. No log entries were returned for node-5/node-6 in the top-20 sample.\n"
    "- error_logs.csv remains empty (row count = 0) in the dataset.\n\n"
    "Interpretation: node-5 shows clear node-level I/O and load abnormalities (highest priority), and shippingservice-0 shows\n"
    "sustained container memory-failure indicators plus intermittent network spikes; traces and logs for shippingservice show\n"
    "normal/low trace latencies and zero logged errors but normal request counts—focus investigation on node I/O and shippingservice\n"
    "container memory/network subsystems."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                           kpi_name fault_type          start_timestamp            end_timestamp  length_minutes  max_value_in_run  min_value_in_run    global_P95    global_P5  severity_score
0              node-5                                    system.io.rkb_s       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31         64.000000          0.000000      0.000000     0.000000       64.000000
1   shippingservice-0  container.node-6.container_memory_failures.hie...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31         33.000000          0.000000      0.000000     0.000000       33.000000
2   shippingservice-0  container.node-6.container_memory_failures.con...       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31         33.000000          0.000000      0.000000     0.000000       33.000000
3              node-5                                  system.io.r_await       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31          1.000000          0.000000      0.000000     0.000000        1.000000
4   shippingservice-0  container.node-6.container_network_receive_MB....       high  2022-03-20 13:50:00 CST  2022-03-20 13:50:00 CST               1          0.705101          0.705101      0.048254     0.034780        0.656848
5              node-5                                      system.load.1       high  2022-03-20 13:36:00 CST  2022-03-20 13:37:00 CST               2          2.590000          1.620000      1.620000     0.170000        0.598765
6              node-5                                      system.io.r_s       high  2022-03-20 13:30:00 CST  2022-03-20 14:00:00 CST              31          0.500000          0.000000      0.000000     0.000000        0.500000
7   shippingservice-0  container.node-6.container_network_transmit_pa...       high  2022-03-20 13:50:00 CST  2022-03-20 13:50:00 CST               1        427.500000        427.500000    292.000000   212.000000        0.464041
8              node-5                                  system.cpu.system       high  2022-03-20 13:38:00 CST  2022-03-20 13:38:00 CST               1          4.590000          4.590000      3.230000     2.410000        0.421053
9              node-5                                      system.load.1       high  2022-03-20 13:31:00 CST  2022-03-20 13:31:00 CST               1          2.080000          2.080000      1.620000     0.170000        0.283951
10  shippingservice-0  container.node-6.container_network_receive_pac...       high  2022-03-20 13:50:00 CST  2022-03-20 13:50:00 CST               1        546.500000        546.500000    444.000000   315.500000        0.230856
11             node-5                               system.cpu.pct_usage       high  2022-03-20 13:38:00 CST  2022-03-20 13:38:00 CST               1          9.710000          9.710000      8.310000     6.190000        0.168472
12             node-5                                      system.load.5       high  2022-03-20 13:36:00 CST  2022-03-20 13:39:00 CST               4          1.360000          1.170000      1.170000     0.330000        0.162393
13             node-5                                  system.io.w_await       high  2022-03-20 13:45:00 CST  2022-03-20 13:46:00 CST               2          0.940000          0.000000      0.790000     0.000000        0.150000
14  shippingservice-0  container.node-6.container_network_transmit_MB...       high  2022-03-20 13:47:00 CST  2022-03-20 13:47:00 CST               1          0.704482          0.704482      0.602880     0.374189        0.101602
15  shippingservice-0  mesh.destination.checkoutservice.shippingservi...       high  2022-03-20 14:00:00 CST  2022-03-20 14:00:00 CST               1       8680.000000       8680.000000   7955.000000  4707.500000        0.091138
16  shippingservice-0  mesh.destination.checkoutservice.shippingservi...       high  2022-03-20 14:00:00 CST  2022-03-20 14:00:00 CST               1         12.000000         12.000000     11.000000     7.000000        0.090909
17  shippingservice-0  mesh.destination.checkoutservice.shippingservi...       high  2022-03-20 14:00:00 CST  2022-03-20 14:00:00 CST               1         12.000000         12.000000     11.000000     7.000000        0.090909
18  shippingservice-0  mesh.destination.checkoutservice.shippingservi...       high  2022-03-20 14:00:00 CST  2022-03-20 14:00:00 CST               1         12.000000         12.000000     11.000000     7.000000        0.090909
19  shippingservice-0  mesh.destination.checkoutservice.shippingservi...       high  2022-03-20 14:00:00 CST  2022-03-20 14:00:00 CST               1      15000.000000      15000.000000  13750.000000  8125.000000        0.090909,                   timestamp            cmdb_id                                trace_name     value  global_P95  global_P5
0   2022-03-20 13:30:00 CST  checkoutservice-0  trace.to_shippingservice-0.duration_mean  0.000074    0.000077   0.000066
1   2022-03-20 13:30:00 CST  checkoutservice-0   trace.to_shippingservice-0.duration_p95  0.000080    0.000088   0.000067
2   2022-03-20 13:30:00 CST  checkoutservice-0     trace.to_shippingservice-0.error_rate  0.000000    0.000000   0.000000
3   2022-03-20 13:30:00 CST  checkoutservice-0      trace.to_shippingservice-0.row_count  3.000000    4.000000   2.000000
4   2022-03-20 13:30:00 CST  checkoutservice-0  trace.to_shippingservice-1.duration_mean  0.000063    0.000092   0.000064
5   2022-03-20 13:30:00 CST  checkoutservice-0   trace.to_shippingservice-1.duration_p95  0.000066    0.000124   0.000066
6   2022-03-20 13:30:00 CST  checkoutservice-0     trace.to_shippingservice-1.error_rate  0.000000    0.000000   0.000000
7   2022-03-20 13:30:00 CST  checkoutservice-0      trace.to_shippingservice-1.row_count  2.000000    4.000000   2.000000
8   2022-03-20 13:30:00 CST  checkoutservice-0  trace.to_shippingservice-2.duration_mean  0.000067    0.000068   0.000059
9   2022-03-20 13:30:00 CST  checkoutservice-0   trace.to_shippingservice-2.duration_p95  0.000069    0.000076   0.000060
10  2022-03-20 13:30:00 CST  checkoutservice-0     trace.to_shippingservice-2.error_rate  0.000000    0.000000   0.000000
11  2022-03-20 13:30:00 CST  checkoutservice-0      trace.to_shippingservice-2.row_count  3.000000    4.000000   2.000000
12  2022-03-20 13:30:00 CST  checkoutservice-1  trace.to_shippingservice-0.duration_mean  0.000069    0.000086   0.000066
13  2022-03-20 13:30:00 CST  checkoutservice-1   trace.to_shippingservice-0.duration_p95  0.000071    0.000103   0.000067
14  2022-03-20 13:30:00 CST  checkoutservice-1     trace.to_shippingservice-0.error_rate  0.000000    0.000000   0.000000
15  2022-03-20 13:30:00 CST  checkoutservice-1      trace.to_shippingservice-0.row_count  3.000000    4.000000   1.450000
16  2022-03-20 13:30:00 CST  checkoutservice-1  trace.to_shippingservice-1.duration_mean  0.000067    0.000074   0.000065
17  2022-03-20 13:30:00 CST  checkoutservice-1   trace.to_shippingservice-1.duration_p95  0.000068    0.000084   0.000066
18  2022-03-20 13:30:00 CST  checkoutservice-1     trace.to_shippingservice-1.error_rate  0.000000    0.000000   0.000000
19  2022-03-20 13:30:00 CST  checkoutservice-1      trace.to_shippingservice-1.row_count  2.000000    4.000000   2.000000,                   timestamp            cmdb_id         log_name  value
0   2022-03-20 13:30:00 CST  shippingservice-0  log.error_count    0.0
1   2022-03-20 13:30:00 CST  shippingservice-0    log.row_count  121.0
2   2022-03-20 13:31:00 CST  shippingservice-0  log.error_count    0.0
3   2022-03-20 13:31:00 CST  shippingservice-0    log.row_count   95.0
4   2022-03-20 13:32:00 CST  shippingservice-0  log.error_count    0.0
5   2022-03-20 13:32:00 CST  shippingservice-0    log.row_count  114.0
6   2022-03-20 13:33:00 CST  shippingservice-0  log.error_count    0.0
7   2022-03-20 13:33:00 CST  shippingservice-0    log.row_count  110.0
8   2022-03-20 13:34:00 CST  shippingservice-0  log.error_count    0.0
9   2022-03-20 13:34:00 CST  shippingservice-0    log.row_count   97.0
10  2022-03-20 13:35:00 CST  shippingservice-0  log.error_count    0.0
11  2022-03-20 13:35:00 CST  shippingservice-0    log.row_count  116.0
12  2022-03-20 13:36:00 CST  shippingservice-0  log.error_count    0.0
13  2022-03-20 13:36:00 CST  shippingservice-0    log.row_count  107.0
14  2022-03-20 13:37:00 CST  shippingservice-0  log.error_count    0.0
15  2022-03-20 13:37:00 CST  shippingservice-0    log.row_count  122.0
16  2022-03-20 13:38:00 CST  shippingservice-0  log.error_count    0.0
17  2022-03-20 13:38:00 CST  shippingservice-0    log.row_count   95.0
18  2022-03-20 13:39:00 CST  shippingservice-0  log.error_count    0.0
19  2022-03-20 13:39:00 CST  shippingservice-0    log.row_count  110.0,    error_logs_row_count
0                     0)```
```